# Juice+ v2: does the way a prospect moves at the Combine show up in NFL games?

**Juice+** puts a player's movement on one scale (100 = position average, 10 points = one standard
deviation). It is measured from Combine tracking and re-measured the same way from regular-season game
tracking (2023 to 2025). This notebook merges the best parts of `juice.py` and the shared notebook.

**What this version does**
- **Everyone counts.** Every player with Combine tracking who has at least one usable game snap gets a
  Game Juice+. There is no minimum snap count. Low-snap players are pulled toward 100, so a handful of
  plays can never produce an extreme score.
- **Like is compared with like.** Plays are compared only with plays at the same position, alignment, route
  (receivers and tight ends), coverage job (DBs) and play length. Snaps in decided games and dead plays are dropped,
  but a player who only has such snaps is kept.
- **Matched on both sides.** Explosiveness is the best effort of speed at 1.0 s and distance at 1.5 s from a
  standstill. Agility counts only where the Combine drills agree with each other.
- **Less noise on the Combine side.** Each tracked piece is averaged with the stopwatch tests of the same trait.
  A player must still have the tracked piece.
- **Weighted by repeatability.** Game pieces are weighted by how repeatable each one is.
- **Checked against more than movement.** Section 8 tests EPA, career starts, pass-rush pressure and the
  NFL's own get-off time.

## 0. Setup and settings

In [ ]:
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Image

warnings.filterwarnings("ignore")

In [ ]:
# Folders. Set JUICE_DATA_DIR to a local copy of the competition files, or let kagglehub download them.
COMP = "nfl-big-data-bowl-2027"
DATA_DIR = os.getenv("JUICE_DATA_DIR")
CACHE = Path(os.getenv("JUICE_CACHE_DIR", "juice_cache"))
OUT = Path(os.getenv("JUICE_OUT_DIR", "juice_nb_output"))
if os.path.isdir("/kaggle/input"):
    hits = glob.glob("/kaggle/input/**/players.csv", recursive=True)
    if hits:
        DATA_DIR = os.path.dirname(hits[0])
    CACHE, OUT = Path("/kaggle/working/juice_cache"), Path("/kaggle/working/juice_nb_output")
for _p in (CACHE, OUT):
    _p.mkdir(parents=True, exist_ok=True)

# Model settings.
SEASONS = (2023, 2024, 2025)
CHUNK_SIZE = 2_000_000
MIN_PLAY_SECONDS = 3.0
MIN_FRAMES = int(round(MIN_PLAY_SECONDS * 10))   # plays shorter than this are dead plays
MIN_FRAMES_OPEN = 20                             # plays used for top speed and agility
BLEND_STOPWATCH = True                           # average tracked pieces with stopwatch tests
DROP_BLOWOUTS, GARBAGE_WP = True, 0.05           # drop snaps when home win probability is under 5% or over 95%
GATE_ALPHA, MIN_PLAYERS_DRILL, MIN_OVERLAP = 0.5, 20, 15
RELIABILITY_MIN = (200, 50)                      # reliabilities are estimated on well-sampled players only
SEASON_MIN = (100, 30)                           # persistence check, one season at a time
BELOW_AVG, ELITE, LOW_JUICE, HIGH_JUICE = 100, 110, 90, 110
HIGH_PICK, LOW_PICK = 64, 140
BOOT = 1000
REUSE_JUICE_PY_CACHE = False   # True only if game_play_table_v2.csv in the cache folder was written by juice.py
MAKE_GIF, PLAYER = False, "Darien Porter"        # Figure 2 re-reads game tracking, so it is off by default
YD_TO_MPH = 3600 / 1760
rng = np.random.default_rng(42)

# Position groups. Linebackers and running backs have too few players with Combine tracking to score.
PGM = {"T": "OL", "G": "OL", "C": "OL", "DT": "DL", "NT": "DL", "DE": "EDGE", "OLB": "EDGE",
       "CB": "DB", "DB": "DB", "SS": "DB", "FS": "DB", "WR": "WR", "TE": "TE"}
ALIAS = {"PASS_PRO-MIRROR_DRILL": "PASS_PRO_MIRROR_DRILL", "SPEED_OUT_LEFT": "SPEED_OUT_ROUTE_LEFT",
         "OVER_THE_SHOULDER_ADJUST": "OVER_SHOULDER_ADJUST"}
KEYS = ["game_id", "play_id", "nfl_id"]

# Colour-blind-safe palette (Okabe-Ito) and readable plot defaults.
OI = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#F0E442", "#000000"]
GROUP_COL = dict(zip(["DB", "DL", "EDGE", "OL", "TE", "WR"], OI))
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})

## 1. Helpers

In [ ]:
_root = None


def get_file(name):
    """Path to a competition file, from JUICE_DATA_DIR or a Kaggle download."""
    global _root
    if DATA_DIR:
        return os.path.join(DATA_DIR, name)
    if _root is None:
        import kagglehub
        _root = kagglehub.competition_download(COMP)
    direct, nested = os.path.join(_root, name), os.path.join(_root, COMP, name)
    return direct if os.path.exists(direct) else nested


def load_csv(name, **kw):
    return pd.read_csv(get_file(name), **kw)


def save(df, name):
    df.to_csv(OUT / name, index=False)


def circ(deg):
    return (deg + 180.0) % 360.0 - 180.0


def zs(s):
    sd = s.std()
    return pd.Series(np.nan, index=s.index) if not np.isfinite(sd) or sd == 0 else (s - s.mean()) / sd


def safe_spearman(x, y):
    p = pd.DataFrame({"x": np.asarray(x, float), "y": np.asarray(y, float)}).dropna()
    if len(p) < 4 or p.x.nunique() < 2 or p.y.nunique() < 2:
        return np.nan
    return float(spearmanr(p.x, p.y)[0])


def fisher_ci(r, n):
    if not np.isfinite(r) or n <= 3 or abs(r) >= 1:
        return np.nan, np.nan
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    return float(np.tanh(z - 1.96 * se)), float(np.tanh(z + 1.96 * se))


def pooled_fisher(rs_ns):
    """Pool correlations across groups by Fisher z weighted by n-3. Returns r, lo, hi."""
    rs_ns = [(r, n) for r, n in rs_ns if np.isfinite(r) and abs(r) < 1 and n > 3]
    if not rs_ns:
        return np.nan, np.nan, np.nan
    w = sum(n - 3 for _, n in rs_ns)
    r = float(np.tanh(sum(np.arctanh(r) * (n - 3) for r, n in rs_ns) / w))
    se = 1.96 / np.sqrt(w)
    return r, float(np.tanh(np.arctanh(r) - se)), float(np.tanh(np.arctanh(r) + se))


def sb(k, r):
    """Spearman-Brown: reliability of the mean of k observations, each with reliability r."""
    k = np.asarray(k, dtype=float)
    if not np.isfinite(r) or r <= 0:
        return np.zeros_like(k, dtype=float)
    return np.clip(k * r / (1 + (k - 1) * r), 0, 1)


def single_obs_reliability(r_full, k):
    """Invert Spearman-Brown: reliability of ONE observation."""
    if not np.isfinite(r_full) or r_full <= 0 or not np.isfinite(k) or k <= 1:
        return 0.0
    r_full = min(float(r_full), 0.999)
    return r_full / (k - (k - 1) * r_full)


def scale100(df, col, by="pgm"):
    g = df.groupby(by)[col]
    return 100 + 10 * (df[col] - g.transform("mean")) / g.transform("std").replace(0, np.nan)


def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n
    den = 1 + z ** 2 / n
    ctr = (p + z ** 2 / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return ctr - half, ctr + half


def auc_ci(y, s, reps=1000, seed=3):
    y, s = np.asarray(y).astype(int), np.asarray(s, dtype=float)
    ok = ~np.isnan(s)
    y, s = y[ok], s[ok]
    if len(y) < 10 or y.min() == y.max():
        return np.nan, np.nan, np.nan
    r, boots = np.random.default_rng(seed), []
    for _ in range(reps):
        i = r.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            boots.append(roc_auc_score(y[i], s[i]))
    lo, hi = np.percentile(boots, [2.5, 97.5]) if boots else (np.nan, np.nan)
    return float(roc_auc_score(y, s)), float(lo), float(hi)


def adjust(df, col, use_class=True, size_adj=True):
    """Leave-one-out residual of `col` within each position group after removing body size
    (weight, weight squared, height) and draft class, standardized to mean 0, sd 1."""
    out = pd.Series(np.nan, index=df.index)
    for _, d in df.groupby("pgm"):
        d = d.dropna(subset=[col, "combine_weight", "combine_height"])
        if len(d) < 15:
            continue
        y = d[col].to_numpy()
        feats = (pd.get_dummies(d.draft_year, prefix="yr", drop_first=True).astype(float)
                 if use_class else pd.DataFrame(index=d.index))
        if size_adj and len(d) >= 25:
            feats["w"] = d.combine_weight.to_numpy()
            feats["h"] = d.combine_height.to_numpy()
            feats["w2"] = ((d.combine_weight - d.combine_weight.mean()) ** 2 / 100).to_numpy()
        if feats.shape[1] == 0:
            pred = np.array([np.delete(y, i).mean() for i in range(len(y))])
        else:
            model = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))
            pred = cross_val_predict(model, feats.to_numpy(), y, cv=LeaveOneOut())
        r = y - pred
        if r.std() > 0:
            out.loc[d.index] = (r - r.mean()) / r.std()
    return out

## 2. Players and Combine tracking

Only players in the six position groups **with Combine tracking** are scored, so the game data covers exactly the
positions the Combine data covers.

In [ ]:
players = load_csv("players.csv")
combine = load_csv("combine_results.csv")
games = load_csv("games.csv")

LOWER_BETTER = [c for c in ["forty", "ten_yd_split", "three_cone", "short_shuttle"] if c in combine.columns]
HIGHER_BETTER = [c for c in ["vertical", "broad_jump", "bench_reps"] if c in combine.columns]
TESTS = combine[["nfl_id"] + LOWER_BETTER + HIGHER_BETTER].copy()

pcols = [c for c in ["nfl_id", "display_name", "nfl_position", "draft_year", "draft_round",
                     "draft_overall_pick"] if c in players.columns]
D = players[pcols].copy()
D["pgm"] = D.nfl_position.map(PGM)
D = D.dropna(subset=["pgm"]).merge(
    combine[["nfl_id", "combine_weight", "combine_height", "ngs_athleticism_score"]], on="nfl_id", how="left")

ct = load_csv("combine_tracking.csv",
              usecols=["nfl_id", "entity_type", "event_id", "drill_name", "drill_type", "time", "x", "y", "s", "a"])
ct = ct[(ct.entity_type == "PLAYER") & ct.nfl_id.notna()].copy()
ct["nfl_id"] = ct.nfl_id.astype(int)
ct = ct[ct.nfl_id.isin(D.nfl_id)]
D = D[D.nfl_id.isin(ct.nfl_id)].copy()
ct["drill_name"] = ct.drill_name.replace(ALIAS)
ct["time"] = pd.to_datetime(ct.time)
ct = ct.sort_values(["event_id", "time"])
ct["t"] = (ct.time - ct.groupby("event_id").time.transform("min")).dt.total_seconds()
print(f"Players with Combine tracking in the six groups: {len(D)}")
print(D.nfl_position.value_counts().to_dict())

## 3. Combine Juice+

Three pieces from the tracked 40-yard dash and the position drills, then one score per player.
- **E (explosiveness):** distance at 1.5 s and speed at 1.0 s.
- **T (top end):** the fastest 10 yards of the run.
- **A (agility):** position drills, used only for groups whose drills agree with each other.

Each piece is adjusted for body size and draft class, then shrunk by how repeatable it is.
Section 3 ends by averaging each piece with the stopwatch tests of the same trait.

In [ ]:
# The tracked 40: explosiveness (d15, v1) and top end (fly10) for each attempt.
def feats40(a):
    s, x, t = a.s.to_numpy(), a.x.to_numpy(), a.t.to_numpy()
    moving = np.where(s >= 1.0)[0]
    if len(moving) < 10:
        return None
    i0 = moving[0]
    s, x, t = s[i0:], x[i0:], t[i0:]
    progress = np.maximum.accumulate(np.abs(x - x[0])) + np.arange(len(x)) * 1e-9
    if progress[-1] < 36:
        return None
    elapsed = t - t[0]
    best = 0.0
    for start in np.arange(5.0, min(progress[-1] - 10, 30.0) + 1e-6, 2.5):
        mask = (progress >= start) & (progress <= start + 10)
        if mask.sum() >= 5:
            best = max(best, float(s[mask].mean()))
    smooth = pd.Series(s).rolling(3, center=True, min_periods=1).mean().to_numpy()
    return pd.Series({
        "d15": np.interp(1.5, elapsed, progress),
        "v1": np.interp(1.0, elapsed, smooth),
        "fly10": best if best > 0 else np.nan,
    })

In [ ]:
# Retest reliability: how well attempt 1 predicts attempt 2, within position.
f40 = ct[ct.drill_name == "FORTY_YARD_DASH"]
a40 = f40.groupby("event_id").apply(feats40).dropna(subset=["fly10"])
a40["nfl_id"] = f40.groupby("event_id").nfl_id.first().reindex(a40.index).astype(int)
a40 = a40.sort_index()
a40["k"] = a40.groupby("nfl_id").cumcount()

w2 = a40[a40.k < 2].pivot(index="nfl_id", columns="k", values=["d15", "v1", "fly10"]).dropna()
w2.columns = [f"{f}_{k}" for f, k in w2.columns]
w2 = w2.reset_index().merge(D[["nfl_id", "pgm", "draft_year"]], on="nfl_id")


def retest_reliability(w):
    """Attempt-to-attempt Spearman WITHIN position after removing class means,
    pooled by Fisher z. Returns (explosiveness reliability, top-end reliability, dict)."""
    out = {}
    for feat in ["d15", "v1", "fly10"]:
        zsum, wsum = 0.0, 0.0
        for _, d in w.groupby("pgm"):
            if len(d) < 15:
                continue
            x0 = d[feat + "_0"] - d.groupby("draft_year")[feat + "_0"].transform("mean")
            x1 = d[feat + "_1"] - d.groupby("draft_year")[feat + "_1"].transform("mean")
            r = spearmanr(x0, x1)[0]
            if np.isfinite(r) and abs(r) < 1:
                zsum += np.arctanh(r) * (len(d) - 3)
                wsum += len(d) - 3
        out[feat] = float(np.tanh(zsum / wsum)) if wsum > 0 else 0.6
    return (out["d15"] + out["v1"]) / 2, out["fly10"], out


REL_E, REL_T, RELI = retest_reliability(w2)
print(f"Combine retest reliability (within position): d15={RELI['d15']:.2f}, "
      f"v1={RELI['v1']:.2f}, fly10={RELI['fly10']:.2f}")

In [ ]:
# One row per player: average over attempts.
P40 = a40.groupby("nfl_id").agg(
    d15=("d15", "mean"), v1=("v1", "mean"), fly10=("fly10", "mean"),
    attempts=("fly10", "size"),
).reset_index()

In [ ]:
# Agility drills. A group's agility counts only if its drills agree with each other (alpha >= 0.5).
def drill_feats(a):
    s, t, acc = a.s.to_numpy(), a.t.to_numpy(), a.a.to_numpy()
    moving = np.where(s >= 1.0)[0]
    if len(moving) < 10:
        return None
    i0, i1 = moving[0], moving[-1]
    if t[i1] - t[i0] < 1.0:
        return None
    return pd.Series({"mean_s": np.nanmean(s[i0:i1 + 1]),
                      "acc95": np.nanpercentile(acc[i0:i1 + 1], 95)})


od = ct[ct.drill_name != "FORTY_YARD_DASH"]
ad = od.groupby("event_id").apply(drill_feats).dropna(subset=["mean_s"])
ad = ad.join(od.groupby("event_id").agg(
    nfl_id=("nfl_id", "first"), drill_type=("drill_type", "first"),
    drill_name=("drill_name", "first"),
))

In [ ]:
DL_SUBS = ["PASS_RUSH_DRILL", "FRONT_START_WAVE", "FOUR_BAG_AGILITY",
           "RUN_AND_CLUB", "BODY_CONTROL", "RUN_THE_HOOP"]
FAM = {
    "DL": ("SKILL_DRILLS_DL", DL_SUBS),
    "EDGE": ("SKILL_DRILLS_DL", DL_SUBS),
    "OL": ("SKILL_DRILLS_OL", ["PASS_RUSH_DROP", "PASS_PRO", "FIVE_YARD_WAVE",
                               "OL_PULL", "SCREEN"]),
    "DB": ("SKILL_DRILLS_DB", ["BACK_PEDAL_AND_90", "BACK_PEDAL_AND_TRANSITION",
                               "BOX_DRILL", "W_DRILL", "TERYL_AUSTIN", "LINE_DRILL"]),
    "WR": ("SKILL_DRILLS_WR", ["SLANT_ROUTE", "DAGGER_ROUTE", "CURL_ROUTE",
                               "SLOT_STRIKE", "COMEBACK", "SLOT_SAIL", "POST_CORNER",
                               "SPEED_OUT", "GO_ROUTE", "RED_ZONE"]),
    "TE": ("SKILL_DRILLS_TE", ["FLAT_ROUTE", "HOOK_ROUTE", "IN_ROUTE",
                               "WHEEL_ROUTE", "CORNER_ROUTE"]),
}
COMMON = ["THREE_CONE_DRILL", "SHORT_SHUTTLE"]

gate, rbar, agil_parts, GATE_ROWS, WIDE = {}, {}, [], [], {}
for grp, (dtype, subs) in FAM.items():
    ids = set(D.loc[D.pgm == grp, "nfl_id"])
    keep = ((ad.drill_type == dtype)
            & ad.drill_name.apply(lambda n: any(s_ in n for s_ in subs))) | ad.drill_name.isin(COMMON)
    sub = ad[keep & ad.nfl_id.isin(ids)]
    pdr = sub.groupby(["nfl_id", "drill_name"])[["mean_s", "acc95"]].mean().reset_index()
    cnt = pdr.groupby("drill_name").nfl_id.nunique()
    pdr = pdr[pdr.drill_name.isin(cnt[cnt >= MIN_PLAYERS_DRILL].index)].copy()
    if pdr.drill_name.nunique() < 2:
        gate[grp], rbar[grp] = False, np.nan
        continue
    for col in ["mean_s", "acc95"]:
        pdr[col + "_z"] = pdr.groupby("drill_name")[col].transform(zs)
    pdr["pace"] = pdr[["mean_s_z", "acc95_z"]].mean(axis=1)
    wide = pdr.pivot(index="nfl_id", columns="drill_name", values="pace")
    cors, cols = [], list(wide.columns)
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            both = wide[[cols[i], cols[j]]].dropna()
            if len(both) >= MIN_OVERLAP:
                cors.append(np.corrcoef(both.iloc[:, 0], both.iloc[:, 1])[0, 1])
    rb = float(np.nanmean(cors)) if cors else np.nan
    kmed = float(wide.notna().sum(axis=1).median())
    alpha = kmed * rb / (1 + (kmed - 1) * rb) if np.isfinite(rb) and kmed >= 2 else np.nan
    gate[grp], rbar[grp] = bool(np.isfinite(alpha) and alpha >= GATE_ALPHA), rb
    GATE_ROWS.append((grp, len(cols), round(rb, 2) if np.isfinite(rb) else rb,
                      kmed, round(alpha, 2) if np.isfinite(alpha) else alpha, gate[grp]))
    WIDE[grp] = wide
    nd = wide.notna().sum(axis=1)
    agil_parts.append(pd.DataFrame({
        "agil": wide.mean(axis=1).where(nd >= 2), "n_drills": nd}).reset_index())
AGL = (pd.concat(agil_parts, ignore_index=True) if agil_parts
       else pd.DataFrame(columns=["nfl_id", "agil", "n_drills"]))
print("Combine agility gate (group, drills, mean r, median drills, alpha, passes):")
for row in GATE_ROWS:
    print("  ", row)

C = D.merge(P40, on="nfl_id", how="left").merge(AGL, on="nfl_id", how="left")

In [ ]:
# Official tests that measure the same trait as each tracked piece (+1 higher is better, -1 lower is better).
OFFICIAL = {
    "E": {"vertical": 1, "broad_jump": 1, "ten_yd_split": -1},
    "T": {"forty": -1},
    "A": {"three_cone": -1, "short_shuttle": -1},
}
TRACKED_COLS = ["d15", "v1", "fly10", "agil"]
OFFICIAL_COLS = sorted({c_ for d_ in OFFICIAL.values() for c_ in d_} & set(TESTS.columns))


def with_tests(df):
    """Same rows as df, plus the official Combine tests."""
    return df.join(TESTS.set_index("nfl_id"), on="nfl_id")


def combine_pieces(z, c, rel_e, rel_t, rbar_map, gate_map, shrink=True, blend=True):
    """E, T, A from standardized columns z. Tracked pieces are shrunk by their reliability;
    with blend=True each is then averaged with the official tests of the same trait."""
    E = pd.concat([z["d15"], z["v1"]], axis=1).mean(axis=1)
    T = z["fly10"]
    A = z["agil"].where(c.pgm.map(gate_map).fillna(False).astype(bool))
    if shrink:
        k = c.attempts.fillna(1).clip(lower=1)
        m = c.n_drills.fillna(1).clip(lower=1)
        rb = c.pgm.map(rbar_map)
        E, T = E * sb(k, rel_e), T * sb(k, rel_t)
        A = A * np.clip(m * rb / (1 + (m - 1) * rb), 0, 1)
    if blend:
        pieces = {}
        for name, tracked in (("E", E), ("T", T), ("A", A)):
            cols = [(col, sg) for col, sg in OFFICIAL[name].items() if col in z]
            if cols:
                off = pd.concat([sg * z[col] for col, sg in cols], axis=1).mean(axis=1)
                tracked = ((tracked + off) / 2).where(off.notna(), tracked)
            pieces[name] = tracked
        E, T, A = pieces["E"], pieces["T"], pieces["A"]
    return E, T, A


def build_scores(Cin, size_adj=True, shrink=True, blend=True):
    c = Cin.copy()
    src = with_tests(Cin)
    z = {col: adjust(src, col, size_adj=size_adj) for col in TRACKED_COLS + (OFFICIAL_COLS if blend else [])}
    for col in TRACKED_COLS:
        c[col + "_z"] = z[col]
    c["E"], c["T"], c["A"] = combine_pieces(z, c, REL_E, REL_T, rbar, gate, shrink, blend)
    c["raw"] = c[["E", "T", "A"]].mean(axis=1, skipna=True)
    c.loc[c["E"].isna() | c["T"].isna(), "raw"] = np.nan   # E and T are required
    for comp in ["E", "T", "A", "raw"]:
        c[comp + "_plus"] = scale100(c, comp)
    c["juice"] = c["raw_plus"]
    return c

In [ ]:
# Build the scores. S is the main score; TRK is tracked-only, kept to show what the blend adds.
S = build_scores(C, blend=BLEND_STOPWATCH)
S["juice_unadjusted"] = build_scores(C, size_adj=False, blend=BLEND_STOPWATCH)["juice"].values
# Tracked-only version, saved so outcome.py can show what the blend adds.
TRK = build_scores(C, blend=False)
S["juice_tracked_only"] = TRK["juice"].values
for comp in ["E", "T", "A"]:
    S[f"{comp}_trk_plus"] = TRK[f"{comp}_plus"].values
FINAL = S[S.E_plus.notna() & S.T_plus.notna()].copy()
FINAL["juice_final"] = FINAL["juice"]
print(f"Players with a Combine Juice+ score: {len(FINAL)}")

## 4. Game tracking

Every regular-season snap in 2023 to 2025 is reduced to a few numbers per player-play: peak speed, peak lateral
acceleration, speed at 1.0 s, distance at 1.5 s and the starting speed. This is the slow step, so the result is cached.
Game tracking is read in chunks and only for the columns needed.

In [ ]:
REG = set(games.loc[games.season_type == "REG", "game_id"])


def summarize_chunk(df):
    cols = KEYS + ["n_after", "peak", "alat_max", "s0", "s10", "d15"]
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=cols)
    df = df.sort_values(KEYS + ["time"])
    snap = df.loc[df["event"] == "ball_snap"].groupby(KEYS).time.min().rename("snap")
    df = df.join(snap, on=KEYS)
    df = df[df.time >= df.snap].copy()
    if len(df) == 0:
        return pd.DataFrame(columns=cols)
    gk = [df.game_id, df.play_id, df.nfl_id]
    df["k"] = df.groupby(KEYS).cumcount()
    om = (circ(df.groupby(KEYS)["dir"].diff()) * np.pi / 180.0) / 0.1  # rad/s
    omg = om.groupby(gk)
    om_s = pd.concat([om, omg.shift(1), omg.shift(-1)], axis=1).mean(axis=1)
    df["alat"] = (df.s * om_s.abs()).where(df.s >= 2.0)
    out = df.groupby(KEYS).agg(
        n_after=("k", "size"), peak=("s", "max"), alat_max=("alat", "max"))
    out["s0"] = df.loc[df.k == 0].set_index(KEYS).s
    out["s10"] = df.loc[df.k == 10].set_index(KEYS).s
    # Distance covered in the first 1.5 s after the snap (same idea as the Combine d15).
    p0 = df.loc[df.k == 0].set_index(KEYS)[["x", "y"]]
    p15 = df.loc[df.k == 15].set_index(KEYS)[["x", "y"]]
    out["d15"] = np.hypot(p15.x - p0.x.reindex(p15.index), p15.y - p0.y.reindex(p15.index))
    return out.reset_index()[cols]


def game_pass():
    parts = []
    for yr in SEASONS:
        name = f"game_tracking_{yr}.csv"
        print(f"  Processing {name} in chunks...")
        left = None
        for chunk in pd.read_csv(
            get_file(name), usecols=KEYS + ["time", "s", "dir", "event", "x", "y"],
            chunksize=CHUNK_SIZE,
        ):
            chunk = chunk[chunk.game_id.isin(REG)]
            if left is not None:
                chunk = pd.concat([left, chunk], ignore_index=True)
            if len(chunk) == 0:
                left = None
                continue
            last = chunk.iloc[-1]
            same = (chunk.game_id == last.game_id) & (chunk.play_id == last.play_id)
            left, body = chunk[same], chunk[~same]
            parts.append(summarize_chunk(body))
        parts.append(summarize_chunk(left))
    out = pd.concat([p for p in parts if len(p)], ignore_index=True)
    return out.drop_duplicates(KEYS)

In [ ]:
cache_name = "game_play_table_v2.csv" if REUSE_JUICE_PY_CACHE else "game_play_table_v3.csv"
cache_file = CACHE / cache_name
if cache_file.exists():
    print(f"Using cached play table: {cache_file}")
    PT0 = pd.read_csv(cache_file)
else:
    PT0 = game_pass()
    PT0.to_csv(cache_file, index=False)
print(f"Player-plays in table (all players): {len(PT0):,}")

## 5. Who is in the game data, and why

Plays are attached to the Combine-tracked players and to each play's role (route, coverage job, alignment,
win probability). Three filters follow, and none of them removes a player:
- **Dead plays** (shorter than 3 s) carry no movement, so they are dropped.
- **Decided games** (home win probability under 5% or over 95%) are dropped *unless the player has no other snaps*.
- **No snap minimum.** One usable snap is enough to get a score. The score is then pulled toward 100 in proportion to how little the player played.

In [ ]:
pp = load_csv(
    "player_play.csv",
    usecols=KEYS + ["lined_up_position", "route_ran", "player_get_off", "coverage_assignment",
                    "pass_rushers_encountered", "pre_snap_home_team_win_probability"],
)
PT_ALL = (
    PT0.merge(games[["game_id", "season"]], on="game_id")
       .merge(D[["nfl_id", "pgm", "nfl_position", "draft_year"]], on="nfl_id")
       .merge(pp, on=KEYS, how="left")
)
ids_any = set(PT_ALL.nfl_id)

# Dead plays out first, so a player is judged on plays that carry movement.
PT = PT_ALL[PT_ALL.n_after >= MIN_FRAMES].copy()
ids_usable = set(PT.nfl_id)

# Decided games out, unless that would remove every snap the player has.
wp = PT.pre_snap_home_team_win_probability
PT["blowout"] = (wp <= GARBAGE_WP) | (wp >= 1 - GARBAGE_WP)
has_clean = (~PT.blowout).groupby(PT.nfl_id).transform("any")
PT["blowout_only_player"] = ~has_clean
if DROP_BLOWOUTS:
    PT = PT[~PT.blowout | PT.blowout_only_player].copy()
print(f"Players who only have decided-game snaps (kept anyway): {PT.loc[PT.blowout_only_player, 'nfl_id'].nunique()}")

PT["role"] = (
    (PT.pgm.isin(["WR", "TE"]) & PT.route_ran.notna())
    | (PT.pgm.isin(["DL", "EDGE"]) & PT.player_get_off.notna())
    | ((PT.pgm == "DB") & PT.coverage_assignment.notna() & (PT.coverage_assignment != "NO_ASSIGNMENT"))
    | ((PT.pgm == "OL") & PT.pass_rushers_encountered.notna())
)
PT["align"] = np.where(PT.pgm.isin(["DL", "EDGE", "DB", "WR"]), PT.lined_up_position.fillna("UNK"), "ALL")
PT["len_s"] = np.minimum(PT.n_after // 10, 10)


def coverage_family(a):
    """Collapse coverage assignments into the jobs a DB actually has. Snaps with no assignment get their own key."""
    if pd.isna(a) or a == "NO_ASSIGNMENT":
        return "NO_JOB"
    a = str(a)
    if a == "MAN":
        return "MAN"
    if a.startswith("DEEP_THIRD") or a == "DEEP_FREE":
        return "DEEP_THIRD"
    if a.startswith("QUARTERS"):
        return "QUARTERS"
    if a.startswith(("DEEP_HALF", "PREVENT")):
        return "DEEP_HALF"
    if a.startswith(("CURL_FLAT", "FLAT", "HOOK_CURL")) or a == "HOLE":
        return "SHORT_ZONE"
    return "OTHER"


# One "route" key for everyone: the route for WR/TE, the coverage job for DBs, nothing for the rest.
PT["route"] = np.where(PT.pgm.isin(["WR", "TE"]), PT.route_ran.fillna("NO_ROUTE"),
                       np.where(PT.pgm == "DB", PT.coverage_assignment.map(coverage_family), "NONE"))
print("DB coverage keys:", PT.loc[PT.pgm == "DB", "route"].value_counts().to_dict())
leftover = PT.loc[(PT.pgm == "DB") & (PT.route == "OTHER"), "coverage_assignment"].value_counts()
print("Assignments still in the OTHER bucket (check these):", leftover.to_dict())

### Roster accounting
Every player with Combine tracking ends up in exactly one row.

In [ ]:
n_tracked = len(D)
rows = [("Players with Combine tracking (six groups)", n_tracked),
        ("  no regular-season game snap in the tracking data", n_tracked - len(ids_any & set(D.nfl_id))),
        ("  only dead plays (under 3 s)", len(ids_any - ids_usable)),
        ("  at least one usable game snap", len(ids_usable))]
ACCOUNT_ROWS = rows
PT_PLAYS = PT.groupby("nfl_id").size()

## 6. Context adjustment and game components

A burst or a top speed means little without context, so each play is compared with plays at the same position,
alignment, route or coverage job, and play length. The player's score is the 95th percentile of those adjusted values.

In [ ]:
def demean(df, col, key_sets, min_cell=100):
    mu = df.groupby(key_sets[-1])[col].transform("mean")
    for keys in reversed(key_sets[:-1]):
        m = df.groupby(keys)[col].transform("mean")
        n = df.groupby(keys)[col].transform("count")
        mu = m.where(n >= min_cell, mu)
    return df[col] - mu


def backfit(df, col, factors, min_cell=100, base=("nfl_position",)):
    base = list(base)
    tmp = df[base + list(factors)].copy()
    tmp["_r"] = demean(df, col, [base], min_cell)
    for _ in range(1 if len(factors) <= 1 else 3):
        for f in factors:
            tmp["_r"] = demean(tmp, "_r", [base + [f], base], min_cell)
    return tmp["_r"]


def burst_mask(P):
    """Plays that are real standing-start bursts in the player's role."""
    return P.role & (P.s0 < 1.0) & P.s10.notna() & P.d15.notna()


def add_adjusted(frame):
    P = frame.copy()
    factors = ["route", "len_s", "align"]
    P["peak_c"], P["alat_c"], P["s10_c"], P["d15_c"], P["expl_c"] = (
        np.nan, np.nan, np.nan, np.nan, np.nan)
    open_play = P.n_after >= MIN_FRAMES_OPEN
    if open_play.any():
        P.loc[open_play, "peak_c"] = backfit(P[open_play], "peak", factors)
        P.loc[open_play, "alat_c"] = backfit(P[open_play], "alat_max", factors)
    ok = burst_mask(P)
    if ok.any():
        cells = [["nfl_position", "align", "route"], ["nfl_position", "align"], ["nfl_position"]]
        P.loc[ok, "s10_c"] = demean(P[ok], "s10", cells)
        P.loc[ok, "d15_c"] = demean(P[ok], "d15", cells)
        # Put the two on one scale (divide by the spread within position group), then average.
        # This mirrors the Combine E piece, which averages distance at 1.5 s and speed at 1.0 s.
        sd_s = P.loc[ok].groupby("pgm").s10_c.transform("std")
        sd_d = P.loc[ok].groupby("pgm").d15_c.transform("std")
        P.loc[ok, "expl_c"] = (P.loc[ok, "s10_c"] / sd_s + P.loc[ok, "d15_c"] / sd_d) / 2
    return P


def q95(s):
    return s.quantile(0.95)

In [ ]:
def game_components(P, min_plays=1, min_bursts=0, strict_join=False):
    """One row per player: top speed, agility, explosiveness and counts (P already adjusted).
    The defaults keep every player with at least one usable play."""
    burst = P[burst_mask(P)]
    gb = burst.groupby("nfl_id").agg(nb=("expl_c", "size"), g_expl=("expl_c", q95))
    open_ = P[P.n_after >= MIN_FRAMES_OPEN]
    ga = open_.groupby("nfl_id").agg(
        npl=("peak_c", "size"), n_alat=("alat_c", "count"),
        g_top=("peak_c", q95), g_agil=("alat_c", q95),
    )
    G = ga.join(gb, how="inner" if strict_join else "left")
    G["nb"] = G.nb.fillna(0)
    G = G[(G.nb >= min_bursts) & (G.npl >= min_plays)].reset_index()
    G = G.merge(
        D[["nfl_id", "display_name", "nfl_position", "pgm", "draft_year", "draft_overall_pick",
           "combine_weight", "combine_height"]], on="nfl_id",
    )
    for c in ["g_top", "g_agil"]:
        G[c] = G[c] - G.groupby("nfl_position")[c].transform("mean")
    return G


PTa = add_adjusted(PT)
G_ALL = game_components(PTa)
print(f"Players with a game score: {len(G_ALL)} of {n_tracked}")
print(f"  with any burst (explosiveness):   {(G_ALL.nb > 0).sum()}")
print(f"  plays per player (median / min):  {G_ALL.npl.median():.0f} / {G_ALL.npl.min():.0f}")

## 7. Reliability, shrinkage and Game Juice+

How repeatable is each game piece? Split each well-sampled player's plays in two random halves and correlate them.
Each piece is then multiplied by `sb(n, r)`, the reliability of an average of n plays (Spearman-Brown). A player with
few plays keeps only a small share of his z-score, so low-snap players sit near 100.
The `confidence` column reports that share, so thin scores can be shown faded or filtered in any chart.
Known limit: the 95th percentile of very few plays is lower on average than that of many plays; shrinkage reduces
but does not fully remove this.

In [ ]:
def split_half(P, ids, col, stat, reps=30, burst=False):
    sub = P[P.nfl_id.isin(ids)]
    if burst:
        sub = sub[burst_mask(sub)]
    else:
        sub = sub[sub.n_after >= MIN_FRAMES_OPEN]
    sub = sub[["nfl_id", col]].dropna().reset_index(drop=True)
    out = []
    for _ in range(reps):
        sub["h"] = rng.integers(0, 2, len(sub))
        g = sub.groupby(["nfl_id", "h"])[col].agg(stat).unstack("h").dropna()
        if len(g) < 10:
            continue
        r = safe_spearman(g[0], g[1])
        if np.isfinite(r) and r > -0.99:
            out.append(2 * r / (1 + r))
    return float(np.nanmean(out)) if out else np.nan

In [ ]:
G_REL = game_components(PTa, *RELIABILITY_MIN, strict_join=True)
if len(G_REL) < 60:
    print("Few well-sampled players; estimating reliabilities on all players instead.")
    G_REL = G_ALL
rel_ids = G_REL.nfl_id
GREL = {
    "top": split_half(PTa, rel_ids, "peak_c", q95),
    "expl": split_half(PTa, rel_ids, "expl_c", q95, burst=True),
    "agil": split_half(PTa, rel_ids, "alat_c", q95),
}
K_TYP = {
    "top": float(G_REL.npl.median()),
    "expl": float(G_REL.nb.replace(0, np.nan).median()),
    "agil": float(G_REL.n_alat.median()),
}
GREL1 = {k: single_obs_reliability(GREL[k], K_TYP[k]) for k in GREL}
GATE_G = bool(np.isfinite(GREL["agil"]) and GREL["agil"] >= 0.5)
print("Split-half reliability (full sample):", {k: round(v, 2) for k, v in GREL.items()})
print("Typical plays per well-sampled player:", {k: round(v) for k, v in K_TYP.items()})
print("Single-play reliability:", {k: round(v, 3) for k, v in GREL1.items()})
print("Game agility included:", GATE_G)

COMP_COL = {"top_z": ("g_top", "npl", "top"),
            "expl_z": ("g_expl", "nb", "expl"),
            "agil_z": ("g_agil", "n_alat", "agil")}
USE = ["top_z", "expl_z"] + (["agil_z"] if GATE_G else [])
_w = {z: max(float(np.nan_to_num(GREL[COMP_COL[z][2]])), 0.0) for z in USE}
_tot = sum(_w.values())
C_WEIGHTS = {z: (w / _tot if _tot > 0 else 1 / len(USE)) for z, w in _w.items()}
print("Game Juice+ component weights:", {k: round(v, 2) for k, v in C_WEIGHTS.items()})

In [ ]:
def c_score(G, how="adjust"):
    """Game Juice+ from a component frame. how='adjust' removes body size and draft class (main score);
    how='zscore' only standardizes within position. `confidence` is the average shrink factor."""
    g = G.copy()
    for z, (col, _, _) in COMP_COL.items():
        g[z] = adjust(g, col) if how == "adjust" else g.groupby("pgm")[col].transform(zs)
    # Combine agility is only scored where its drills agree, so game agility is left out there too.
    g.loc[~g.pgm.map(gate).fillna(False).astype(bool), "agil_z"] = np.nan
    num, den, conf = 0, 0, 0
    for z in USE:
        _, ncol, key = COMP_COL[z]
        k = pd.Series(sb(g[ncol].fillna(0), GREL1[key]), index=g.index)
        shrunk = g[z] * k
        num = num + shrunk.fillna(0) * C_WEIGHTS[z]
        den = den + shrunk.notna() * C_WEIGHTS[z]
        conf = conf + k.where(shrunk.notna(), 0) * C_WEIGHTS[z]
    g["raw"] = num / den.replace(0, np.nan)
    g["confidence"] = conf / den.replace(0, np.nan)
    g["game_juice"] = scale100(g, "raw")
    for z, name in [("top_z", "g_top_plus"), ("expl_z", "g_expl_plus"), ("agil_z", "g_agil_plus")]:
        g[name] = 100 + 10 * g[z]
    return g


GAME = c_score(G_ALL)
GAME = GAME[GAME.game_juice.notna()].copy()
print(f"Game Juice+ scored for {len(GAME)} players; median confidence {GAME.confidence.median():.2f}")
save(GAME.drop(columns=["raw"]), "game_juice_scores.csv")

In [ ]:
# Accounting table, final row.
ACCOUNT_ROWS.append(("  scored (Game Juice+)", len(GAME)))
ACCOUNT = pd.DataFrame(ACCOUNT_ROWS, columns=["group", "players"])
ACCOUNT["share_of_tracked"] = (100 * ACCOUNT.players / n_tracked).round(1)
display(ACCOUNT)
save(ACCOUNT, "roster_accounting.csv")
print("Scored players by number of usable plays:")
print(pd.cut(GAME.npl, [0, 9, 49, 199, 100000], labels=["1-9", "10-49", "50-199", "200+"]).value_counts().sort_index().to_dict())
by_pos = D.groupby("pgm").nfl_id.nunique().to_frame("tracked").join(GAME.groupby("pgm").nfl_id.nunique().rename("scored"))
display(by_pos)

In [ ]:
# Season-to-season persistence of Game Juice+: the ceiling for any Combine link.
persist_rows, season_scores = [], []
for seas, sub in PT.groupby("season"):
    Gs = game_components(add_adjusted(sub), *SEASON_MIN, strict_join=True)
    if len(Gs) < 10:
        continue
    Gs = c_score(Gs, how="zscore")
    Gs["season"] = seas
    season_scores.append(Gs[["nfl_id", "season", "raw"]])
if season_scores:
    wide_s = pd.concat(season_scores).pivot(index="nfl_id", columns="season", values="raw")
    cols_s = sorted(wide_s.columns)
    for a_, b_ in zip(cols_s[:-1], cols_s[1:]):
        x = wide_s[[a_, b_]].dropna()
        if len(x) >= 10:
            persist_rows.append({"season_pair": f"{a_}->{b_}", "n_players": len(x),
                                 "spearman": safe_spearman(x[a_], x[b_])})
persist_df = pd.DataFrame(persist_rows)
display(persist_df)

## 8. Does Combine Juice+ predict Game Juice+?

The test: within each position group, do players with a higher Combine Juice+ also have a higher Game Juice+?
Everything is reported with 95% intervals. Two caveats up front.
- The Combine formula was tuned on these same players, so most numbers below are in-sample. The two exceptions are the
  **held-out formula** (each draft class scored by a formula fit without it) and the **out-of-class R-squared**.
- Players with few snaps are pulled toward 100 by design, which dilutes a correlation. Each result is therefore also shown
  for players with 200+ plays.

In [ ]:
# Score bands and the stopwatch comparison scores (RAS-style is a percentile average of the official tests).
BAND_EDGES = [-np.inf, LOW_JUICE, BELOW_AVG, ELITE, np.inf]
BAND_LABELS = ["Low (<90)", "Below avg (90-100)", "Above avg (100-110)", "Elite (110+)"]

# --- comparison frame: Combine tracking score vs stopwatch scores -----------------
CMP = FINAL.merge(TESTS, on="nfl_id", how="left")


def ras_style(df):
    out = pd.Series(np.nan, index=df.index)
    for _, d in df.groupby("pgm"):
        pct = {c_: d[c_].rank(pct=True, ascending=(c_ not in LOWER_BETTER))
               for c_ in LOWER_BETTER + HIGHER_BETTER
               if c_ in d.columns and d[c_].notna().sum() >= 15}
        if pct:
            Pp = pd.DataFrame(pct)
            out.loc[d.index] = (10 * Pp.mean(axis=1)).where(Pp.notna().sum(axis=1) >= 4)
    return out


def adj_or_nan(df, col, sign=1):
    return sign * adjust(df, col) if col in df.columns else pd.Series(np.nan, index=df.index)


CMP["ras_style"] = ras_style(CMP)
CMP["ngs_adj"] = adj_or_nan(CMP, "ngs_athleticism_score")
CMP["ras_adj"] = adj_or_nan(CMP, "ras_style")
CMP["forty_adj"] = adj_or_nan(CMP, "forty", -1)
CMP["split_adj"] = adj_or_nan(CMP, "ten_yd_split", -1)
CMP["forty_neg"] = -CMP["forty"] if "forty" in CMP.columns else np.nan

In [ ]:
# Fully held-out Combine formula: each draft class is scored by a formula fitted without it.
def gate_fn(ids):
    gt, rb_ = {g: False for g in FAM}, {g: np.nan for g in FAM}
    for g, wide in WIDE.items():
        w = wide[wide.index.isin(ids)]
        cors = []
        for i in range(w.shape[1]):
            for j in range(i + 1, w.shape[1]):
                both = w.iloc[:, [i, j]].dropna()
                if len(both) >= MIN_OVERLAP:
                    cors.append(np.corrcoef(both.iloc[:, 0], both.iloc[:, 1])[0, 1])
        rb = float(np.nanmean(cors)) if cors else np.nan
        k = float(w.notna().sum(axis=1).median()) if len(w) else 0
        alpha = k * rb / (1 + (k - 1) * rb) if np.isfinite(rb) and k >= 2 else np.nan
        gt[g], rb_[g] = bool(np.isfinite(alpha) and alpha >= GATE_ALPHA), rb
    return gt, rb_


def adjust_frozen(df, col, train):
    """Size model fit on the training classes only; the held-out class is standardized
    against itself."""
    out = pd.Series(np.nan, index=df.index)
    for _, d in df.groupby("pgm"):
        d = d.dropna(subset=[col, "combine_weight", "combine_height"])
        tr, te = d[train.loc[d.index]], d[~train.loc[d.index]]
        if len(tr) < 15 or len(te) < 3:
            continue

        def feats(x):
            return np.column_stack([x.combine_weight, x.combine_height,
                                    (x.combine_weight - tr.combine_weight.mean()) ** 2 / 100])
        if len(tr) >= 25:
            mdl = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20))).fit(feats(tr), tr[col])
            p = mdl.predict(feats(te))
        else:
            p = tr[col].mean()
        r = te[col].values - p
        if np.std(r) > 0:
            out.loc[te.index] = (r - np.mean(r)) / np.std(r)
    return out


JF = pd.Series(np.nan, index=C.index)
CT = with_tests(C)
for cls in sorted(C.draft_year.dropna().unique()):
    train = C.draft_year != cls
    ids_tr = set(C.nfl_id[train])
    RE_f, RT_f, _ = retest_reliability(w2[w2.nfl_id.isin(ids_tr)])
    gate_f, rbar_f = gate_fn(ids_tr)
    z = {col: adjust_frozen(CT, col, train)
         for col in TRACKED_COLS + (OFFICIAL_COLS if BLEND_STOPWATCH else [])}
    E, T, A = combine_pieces(z, C, RE_f, RT_f, rbar_f, gate_f, blend=BLEND_STOPWATCH)
    raw = pd.concat([E, T, A], axis=1).mean(axis=1).where(E.notna() & T.notna())
    te = ~train & raw.notna()
    g_ = raw[te].groupby(C.pgm[te])
    JF[te] = 100 + 10 * (raw[te] - g_.transform("mean")) / g_.transform("std")
JF_MAP = dict(zip(C.nfl_id[JF.notna()], JF[JF.notna()]))

In [ ]:
# One row per player with both scores.
M = FINAL[["nfl_id", "juice_final", "E_plus", "T_plus", "A_plus"]].merge(
    GAME[["nfl_id", "display_name", "nfl_position", "pgm", "draft_year", "game_juice", "g_top_plus",
          "g_expl_plus", "g_agil_plus", "npl", "nb", "confidence"]], on="nfl_id")
M["g_below"] = M.game_juice < BELOW_AVG
M["g_above"] = M.game_juice >= BELOW_AVG
M["g_elite"] = M.game_juice >= ELITE
M["c_band"] = pd.cut(M.juice_final, BAND_EDGES, labels=BAND_LABELS, right=False)
M["c_p"] = M.groupby("pgm").juice_final.rank(pct=True)
print(f"Players with both scores: {len(M)}  (Combine: {len(FINAL)}, Game: {len(GAME)})")
M_EST = M[M.npl >= RELIABILITY_MIN[0]].copy()
print(f"Established players (200+ plays): {len(M_EST)}")

In [ ]:
def risk_ratio(g, o, reps=2000, seed=7):
    g, o = np.asarray(g, bool), np.asarray(o, bool)
    if g.sum() == 0 or o.mean() == 0:
        return np.nan, (np.nan, np.nan)
    r_ = np.random.default_rng(seed)
    est, boots = o[g].mean() / o.mean(), []
    for _ in range(reps):
        i = r_.integers(0, len(o), len(o))
        gi, oi = g[i], o[i]
        if gi.sum() and oi.mean() > 0:
            boots.append(oi[gi].mean() / oi.mean())
    lo, hi = np.percentile(boots, [2.5, 97.5]) if boots else (np.nan, np.nan)
    return float(est), (float(lo), float(hi))


def risk_answer(label, group_mask, outcome):
    g, o = np.asarray(group_mask, bool), np.asarray(outcome, bool)
    n, k = int(g.sum()), int(o[g].sum())
    rr, (rlo, rhi) = risk_ratio(g, o)
    wlo, whi = wilson(k, n)
    return {"claim": label, "k": k, "n": n, "pct": 100 * k / n if n else np.nan,
            "wilson_lo": 100 * wlo, "wilson_hi": 100 * whi,
            "base_pct": 100 * o.mean(), "risk_ratio": rr, "rr_lo": rlo, "rr_hi": rhi}


def head_to_head(M):
    """Juice+ vs NGS and RAS-style scores on the same players, raw and size-adjusted."""
    if "ngs_athleticism_score" not in CMP.columns or CMP.ras_style.notna().sum() < 30:
        return None
    X = CMP[CMP.ngs_athleticism_score.notna() & CMP.ras_style.notna()].copy()
    weight_r = {g: {s_: spearmanr(d.combine_weight, d[c_], nan_policy="omit")[0] for s_, c_ in
                    [("Juice+", "juice_final"), ("NGS", "ngs_athleticism_score"), ("RAS", "ras_style")]}
                for g, d in X.groupby("pgm") if len(d) >= 15}
    V = (M[["nfl_id", "pgm", "juice_final", "game_juice", "g_top_plus", "T_plus"]]
         .merge(CMP[["nfl_id", "ngs_athleticism_score", "ras_style", "ngs_adj", "ras_adj",
                     "forty_adj", "split_adj", "forty_neg", "juice_unadjusted"]], on="nfl_id", how="left"))
    V = V.dropna(subset=["ngs_athleticism_score", "ras_style", "ngs_adj", "ras_adj", "juice_unadjusted"])
    if len(V) < 30:
        return None

    def pooled_rho(df, col, outcome="game_juice"):
        return spearmanr(df.groupby("pgm")[col].rank(pct=True),
                         df.groupby("pgm")[outcome].rank(pct=True), nan_policy="omit")[0]

    def corr_row(df, col, outcome):
        row = {"ALL": pooled_rho(df, col, outcome)}
        for g, d in df.groupby("pgm"):
            if len(d) >= 15:
                row[g] = spearmanr(d[col], d[outcome], nan_policy="omit")[0]
        return row

    sc = {"Juice+": "juice_final", "Juice+ (no size adj.)": "juice_unadjusted",
          "NGS": "ngs_athleticism_score", "NGS (size adj.)": "ngs_adj",
          "RAS-style": "ras_style", "RAS-style (size adj.)": "ras_adj"}
    ngs_tab = pd.DataFrame({k: corr_row(V, c_, "game_juice") for k, c_ in sc.items()}).T
    U = V.dropna(subset=["T_plus", "forty_adj", "split_adj", "forty_neg"])
    top_tab = (pd.DataFrame({k: corr_row(U, c_, "g_top_plus") for k, c_ in
                             {"Tracked fastest 10": "T_plus", "Official 40 (size adj.)": "forty_adj",
                              "10-yd split (size adj.)": "split_adj", "Official 40 (raw)": "forty_neg"}.items()}).T
               if len(U) >= 15 else pd.DataFrame())
    rng_b, boots = np.random.default_rng(1), {"NGS": [], "RAS": []}
    for _ in range(1000):
        b = V.iloc[rng_b.integers(0, len(V), len(V))]
        j_ = pooled_rho(b, "juice_final")
        boots["NGS"].append(j_ - pooled_rho(b, "ngs_athleticism_score"))
        boots["RAS"].append(j_ - pooled_rho(b, "ras_style"))
    edge = {k: (ngs_tab.loc["Juice+", "ALL"] - ngs_tab.loc[n_, "ALL"], *np.percentile(v, [2.5, 97.5]))
            for k, v, n_ in [("NGS", boots["NGS"], "NGS"), ("RAS", boots["RAS"], "RAS-style")]}
    # When the scores disagree
    for col, nm in [("juice_final", "j_p"), ("ngs_athleticism_score", "n_p"), ("ras_style", "r_p")]:
        X[nm] = 100 * X.groupby("pgm")[col].rank(pct=True)
    X["gap"] = X.j_p - (X.n_p + X.r_p) / 2
    Y = X.merge(M[["nfl_id", "game_juice"]], on="nfl_id")
    Y["g_p"] = 100 * Y.groupby("pgm").game_juice.rank(pct=True)
    disagree = {k: (int(m.sum()), float(Y.g_p[m].mean()) if m.sum() else np.nan) for k, m in
                [("Juice+ higher by 20+ pct pts", Y.gap >= 20), ("agree (within 20)", Y.gap.abs() < 20),
                 ("Juice+ lower by 20+ pct pts", Y.gap <= -20)]}
    return {"n": len(V), "ngs_tab": ngs_tab, "top_tab": top_tab, "edge": edge,
            "weight_r": weight_r, "disagree": disagree, "V": V}


def incremental_and_frozen(M):
    """Out-of-class R-squared (train on two classes, predict the third), with the in-sample
    Combine Juice+ and with the fully held-out formula."""
    keep = ["nfl_id", "draft_overall_pick", "combine_weight", "combine_height"] + [
        c_ for c_ in ["ngs_athleticism_score"] + LOWER_BETTER + HIGHER_BETTER if c_ in CMP.columns]
    IV = M[["nfl_id", "pgm", "draft_year", "juice_final", "game_juice", "npl"]].merge(
        CMP[keep], on="nfl_id", how="left")

    def pct_feature(col, lower_better=False):
        return IV.groupby("pgm")[col].rank(pct=True, ascending=not lower_better).fillna(0.5)

    trad = {c_: pct_feature(c_, c_ in LOWER_BETTER) for c_ in LOWER_BETTER + HIGHER_BETTER if c_ in IV}
    if "ngs_athleticism_score" in IV:
        trad["ngs"] = pct_feature("ngs_athleticism_score")
    trad["weight"] = IV.groupby("pgm").combine_weight.transform(zs).fillna(0)
    trad["height"] = IV.groupby("pgm").combine_height.transform(zs).fillna(0)
    X_trad = pd.DataFrame(trad)
    X_pick = pd.DataFrame({"log_pick": np.log(IV.draft_overall_pick.fillna(260))})
    X_juice = pd.DataFrame({"juice": (IV.juice_final - 100) / 10})
    X_jf = pd.DataFrame({"juice": (IV.nfl_id.map(JF_MAP).fillna(100) - 100) / 10})
    models = {"pick": X_pick, "trad": X_trad, "trad+pick": X_trad.join(X_pick),
              "juice": X_juice, "trad+pick+juice": X_trad.join(X_pick).join(X_juice),
              "juice_frozen": X_jf, "trad+pick+juice_frozen": X_trad.join(X_pick).join(X_jf)}
    yv, cls_ = IV.game_juice.values, IV.draft_year.values

    def out_of_class(Xm):
        pred, base = np.full(len(IV), np.nan), np.full(len(IV), np.nan)
        for c_ in np.unique(cls_):
            tr, te = cls_ != c_, cls_ == c_
            if tr.sum() < 20 or te.sum() == 0:
                continue
            m_ = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 3, 20))).fit(Xm.values[tr], yv[tr])
            pred[te], base[te] = m_.predict(Xm.values[te]), yv[tr].mean()
        return pred, base

    def r2(y_, p, b):
        ok = ~np.isnan(p)
        den = ((y_[ok] - b[ok]) ** 2).sum()
        return 1 - ((y_[ok] - p[ok]) ** 2).sum() / den if den > 0 else np.nan

    oof = {k: out_of_class(Xm) for k, Xm in models.items()}
    R2 = {k: r2(yv, *v) for k, v in oof.items()}

    def gain(full, seed):
        r_, dg = np.random.default_rng(seed), []
        for _ in range(2000):
            i = r_.integers(0, len(yv), len(yv))
            dg.append(r2(yv[i], oof[full][0][i], oof[full][1][i])
                      - r2(yv[i], oof["trad+pick"][0][i], oof["trad+pick"][1][i]))
        return (R2[full] - R2["trad+pick"], *np.nanpercentile(dg, [2.5, 97.5]))

    MF = M[["nfl_id", "pgm", "draft_year", "game_juice", "g_above"]].assign(
        juice_frozen=M.nfl_id.map(JF_MAP)).dropna(subset=["juice_frozen"])
    by_cls = {int(y): safe_spearman(d.juice_frozen, d.game_juice)
              for y, d in MF.groupby("draft_year") if len(d) >= 15}
    f_r, f_lo, f_hi = pooled_fisher(
        [(safe_spearman(d.juice_frozen, d.game_juice), len(d)) for _, d in MF.groupby("draft_year") if len(d) >= 15])
    f_auc = auc_ci(MF.g_above, MF.groupby("pgm").juice_frozen.rank(pct=True)) if len(MF) else (np.nan,) * 3
    return {"R2": R2, "gain": gain("trad+pick+juice", 5), "gain_frozen": gain("trad+pick+juice_frozen", 6),
            "frozen_by_class": by_cls, "frozen_r": (f_r, f_lo, f_hi), "frozen_auc": f_auc, "n_frozen": len(MF)}

In [ ]:
def corr_table(Mx, label):
    rows = []
    for grp, d in list(Mx.groupby("pgm")) + [("ALL", Mx)]:
        if len(d) < 10:
            continue
        r_ = safe_spearman(d.juice_final, d.game_juice)
        lo, hi = fisher_ci(r_, len(d))
        rows.append({"sample": label, "position_group": grp, "n_players": len(d),
                     "spearman": r_, "ci95_low": lo, "ci95_high": hi})
    return pd.DataFrame(rows)


def band_table(Mx):
    return pd.DataFrame({
        "band": BAND_LABELS,
        "n": [int((Mx.c_band == b).sum()) for b in BAND_LABELS],
        "pct_below_avg": [100 * Mx.loc[Mx.c_band == b, "g_below"].mean() for b in BAND_LABELS],
        "pct_above_avg": [100 * Mx.loc[Mx.c_band == b, "g_above"].mean() for b in BAND_LABELS],
        "pct_elite": [100 * Mx.loc[Mx.c_band == b, "g_elite"].mean() for b in BAND_LABELS],
        "mean_game_juice": [Mx.loc[Mx.c_band == b, "game_juice"].mean() for b in BAND_LABELS]})


def answers_table(Mx):
    g_ = Mx.juice_final
    return pd.DataFrame([
        risk_answer("Combine < 100 -> below average in games", g_ < BELOW_AVG, Mx.g_below),
        risk_answer("Combine < 90 -> below average in games", g_ < LOW_JUICE, Mx.g_below),
        risk_answer("Combine >= 110 -> above average in games", g_ >= ELITE, Mx.g_above),
        risk_answer("Combine >= 110 -> elite (110+) in games", g_ >= ELITE, Mx.g_elite),
        risk_answer("Combine >= 100 -> above average in games", g_ >= BELOW_AVG, Mx.g_above)])


def piece_matrix(Mx):
    cm = pd.DataFrame(index=["T (top end)", "A (agility)", "E (explosiveness)"],
                      columns=["game top speed", "game agility", "game explosiveness"], dtype=float)
    for i, cc in enumerate(["T_plus", "A_plus", "E_plus"]):
        for j, gc in enumerate(["g_top_plus", "g_agil_plus", "g_expl_plus"]):
            d = Mx[[cc, gc]].dropna()
            if len(d) >= 15:
                cm.iloc[i, j] = safe_spearman(d[cc], d[gc])
    return cm


STICK = pd.concat([corr_table(M, "everyone scored"), corr_table(M_EST, "200+ plays")], ignore_index=True)
CLASS_TAB = pd.DataFrame([{"draft_year": int(y), "n_players": len(d), "spearman": safe_spearman(d.juice_final, d.game_juice)}
                          for y, d in M.groupby("draft_year") if len(d) >= 15])
BANDS, ANSWERS, CMAT = band_table(M), answers_table(M), piece_matrix(M)
AUC = pd.DataFrame([dict(zip(["auc", "lo", "hi"], auc_ci(M.g_above, M.c_p)), score="percentile within position")])
H2H = head_to_head(M)
INC = incremental_and_frozen(M)

### 8a. Combine vs game, by position group

In [ ]:
display(STICK.round(3))
display(CLASS_TAB.round(3))

### 8b. What does a Combine score say about a player's game movement?

In [ ]:
display(BANDS.round(1))
display(ANSWERS.round(2))
display(AUC.round(3))

### 8c. Which Combine piece matches which game piece?
Rows are Combine pieces, columns are game pieces. A matched piece (T with top speed, E with explosiveness, A with agility)
should be the brightest cell in its row.

In [ ]:
display(CMAT.round(2))

### 8d. Head-to-head with other stopwatch scores
Same players, same game score. An edge counts only if its interval excludes zero.

In [ ]:
if H2H:
    print(f"Same {H2H['n']} players. Rank correlation with Game Juice+:")
    display(H2H["ngs_tab"].round(2))
    if len(H2H["top_tab"]):
        print("Top-end piece vs game top speed:")
        display(H2H["top_tab"].round(2))
    print("Juice+ edge (difference, 95% bootstrap interval):",
          {k: tuple(round(x, 2) for x in v) for k, v in H2H["edge"].items()})
    print("When the scores disagree (n, mean game percentile):",
          {k: (n, round(m, 1) if np.isfinite(m) else m) for k, (n, m) in H2H["disagree"].items()})
else:
    print("NGS or stopwatch columns are missing, so no head-to-head.")

### 8e. The honest tests: held-out formula and out-of-class R-squared

In [ ]:
print("Held-out formula, correlation by draft class:", {k: round(v, 2) for k, v in INC["frozen_by_class"].items()})
print("  pooled:", tuple(round(x, 3) for x in INC["frozen_r"]), f"(n = {INC['n_frozen']})")
print("  AUC (above-average mover):", tuple(round(x, 3) for x in INC["frozen_auc"]))
print("Out-of-class R-squared:", {k: round(v, 3) for k, v in INC["R2"].items()})
print("Gain of Juice+ over draft pick + stopwatch tests (R2, 95% interval):", tuple(round(x, 3) for x in INC["gain"]))
print("  same, with the held-out formula:", tuple(round(x, 3) for x in INC["gain_frozen"]))

## 9. Does it predict anything beyond movement?

Game Juice+ is a movement score built from the same kind of data. These outcomes are different evidence:
- **Career:** share of available games started, ranked within draft class and position.
- **EPA:** expected points added on the player's snaps (flipped for defenders). Raw and adjusted for team and season.
- **Pass rush and pass protection:** pressure rate, disruption rate, pressure held, and the NFL's get-off time.
- **Coverage:** EPA allowed on snaps with a coverage job.

Every score is compared as a within-position percentile. Each row also shows the correlation after removing draft
pick, which answers "does Juice+ tell us something the draft did not?". Intervals are 95% bootstrap intervals.
An interval that includes 0 means no clear link.

In [ ]:
# Combine and game scores for each player, plus career outcomes and stopwatch tests.
GG = GAME[["nfl_id", "game_juice", "g_top_plus", "g_expl_plus"]]
J = FINAL[["nfl_id", "display_name", "nfl_position", "pgm", "draft_year", "juice_final", "E_plus", "T_plus", "A_plus",
           "juice_tracked_only", "E_trk_plus", "T_trk_plus", "A_trk_plus"]]
CR = combine[["nfl_id", "combine_weight", "forty", "ten_yd_split", "vertical", "broad_jump", "three_cone",
              "short_shuttle", "ngs_athleticism_score"]]
PL = players[["nfl_id", "draft_overall_pick"]]
CS = load_csv("player_career_successes.csv")
OD = (J.merge(GG, on="nfl_id", how="left").merge(CR, on="nfl_id", how="left")
       .merge(PL, on="nfl_id", how="left").merge(CS, on="nfl_id", how="left"))
rng_o = np.random.default_rng(0)
report_lines = []


def say(text=""):
    report_lines.append(str(text))
    print(text)


# Career outcome: share of available games started (17 per season since the draft).
OD["seasons"] = 2026 - OD.draft_year
OD["start_share"] = OD.career_games_started / (17 * OD.seasons)
OD["honors"] = (OD.ap_all_pro_1st_team + OD.ap_all_pro_2nd_team + OD.pro_bowl_original_ballot) > 0

In [ ]:
# Play-level outcomes from player_play, regular season only, penalty-nullified plays removed.
# The merge with OD keeps only players with Combine tracking, so positions match the Combine data.
PP_COLS = ["game_id", "nfl_id", "team_abbr", "expected_points_added", "play_nullified_by_penalty", "target",
           "pass_result", "coverage_assignment", "player_get_off", "pressure_allowed", "sack",
           "tackle_for_loss", "time_to_pressure", "time_to_qb_hurry"]
ppo = pd.read_csv(get_file("player_play.csv"), usecols=PP_COLS)
ppo = ppo.merge(OD[["nfl_id", "pgm"]], on="nfl_id").merge(games[["game_id", "season", "season_type"]], on="game_id")
ppo = ppo[(ppo.season_type == "REG") & (ppo.play_nullified_by_penalty.astype(str) != "Y")].copy()


def tf(s):
    """True/False, 1.0/0.0 or 'True'/'False' to 1.0/0.0; anything else to NaN."""
    return s.astype(str).str.lower().map({"true": 1.0, "false": 0.0, "1.0": 1.0, "0.0": 0.0, "1": 1.0, "0": 0.0})


# EPA is the offense's EPA, so it is flipped for defenders. Team codes are used as given.
defense = ppo.pgm.isin(["DL", "EDGE", "DB"])
ppo["epa"] = ppo.expected_points_added.where(~defense, -ppo.expected_points_added)
ppo["side"] = np.where(defense, "D", "O")
ppo["team"] = ppo.team_abbr
ppo["epa_adj"] = ppo.epa - ppo.groupby(["team", "season", "side"]).epa.transform("mean")
rush = ppo.pgm.isin(["DL", "EDGE"]) & ppo.player_get_off.notna() & ppo.pass_result.notna()
cover = ((ppo.pgm == "DB") & ppo.coverage_assignment.notna() & (ppo.coverage_assignment != "NO_ASSIGNMENT")
         & ppo.pass_result.notna())
pass_pro = (ppo.pgm == "OL") & ppo.pressure_allowed.notna()
targeted = ppo.pgm.isin(["WR", "TE"]) & (tf(ppo.target) == 1)
ppo["pressure"] = ppo.time_to_pressure.notna().astype(float)
ppo["disrupt"] = ((tf(ppo.sack) == 1) | (tf(ppo.tackle_for_loss) == 1)
                 | ppo.time_to_pressure.notna() | ppo.time_to_qb_hurry.notna()).astype(float)
ppo["fast_get_off"] = -ppo.player_get_off
ppo["pressure_held"] = 1 - tf(ppo.pressure_allowed)


def per_player(mask, col, min_n):
    s = ppo.loc[mask].groupby("nfl_id")[col].agg(["mean", "count"])
    return s["mean"].where(s["count"] >= min_n)

In [ ]:
everyone = ppo.epa.notna()
OC = pd.DataFrame({
    "epa_field": per_player(everyone, "epa", 200),
    "epa_field_adj": per_player(everyone, "epa_adj", 200),
    "epa_target": per_player(targeted, "epa", 25),
    "epa_coverage": per_player(cover, "epa", 100),
    "pressure_rate": per_player(rush, "pressure", 100),
    "disrupt_rate": per_player(rush, "disrupt", 100),
    "pressure_held": per_player(pass_pro, "pressure_held", 100),
    "get_off_speed": per_player(rush, "fast_get_off", 100),
})
OD = OD.join(OC, on="nfl_id")

# Data check 2: does each outcome look sensible? (rates and counts so definitions can be eyeballed)
say("\nDATA CHECK. player_play rows kept: " + str(len(ppo)) + "; rows per player: "
    + str(ppo.groupby("nfl_id").size().describe().round(0).to_dict()))
say(f"  pass-rush pressure rate (DL/EDGE): {ppo.loc[rush, 'pressure'].mean():.3f}"
    f"   disruption rate: {ppo.loc[rush, 'disrupt'].mean():.3f}"
    f"   OL pressure-held rate: {ppo.loc[pass_pro, 'pressure_held'].mean():.3f}"
    f"   mean EPA on targeted plays (WR/TE): {ppo.loc[targeted, 'epa'].mean():+.3f}")
say("  players with each outcome: " + str(OC.notna().sum().to_dict()))
say(f"  mean get-off time on pass-rush snaps (seconds, lower is faster): {ppo.loc[rush, 'player_get_off'].mean():.3f}")

In [ ]:
# Draft pick, best pick = highest percentile within position group. A pick of 0 or missing = undrafted.
OD["pick"] = OD.draft_overall_pick.where(OD.draft_overall_pick > 0, 260)
OD["pick_p"] = OD.groupby("pgm").pick.rank(pct=True, ascending=False)

# Stopwatch comparison scores.
OD["neg_forty"] = -OD.forty
OD["speed_score"] = OD.combine_weight * 200 / OD.forty ** 4


SCORES = [
    ("Juice+ (as built)", "juice_final"),
    ("Juice+ tracked-only", "juice_tracked_only"),
    ("E piece (as built)", "E_plus"),
    ("E piece tracked-only", "E_trk_plus"),
    ("T piece (as built)", "T_plus"),
    ("T piece tracked-only", "T_trk_plus"),
    ("A piece (as built)", "A_plus"),
    ("A piece tracked-only", "A_trk_plus"),
    ("NGS athleticism score", "ngs_athleticism_score"),
    ("Official 40 (faster = better)", "neg_forty"),
    ("Speed score (weight x 40)", "speed_score"),
    ("Draft pick", "pick_p"),
    ("Game Juice+ (same players)", "game_juice"),
]
GAME_COLS = {"game_juice", "g_top_plus", "g_expl_plus"}
OUTCOMES = [
    ("Career: share of games started (within class)", "start_share"),
    ("EPA on field (good for player's side)", "epa_field"),
    ("EPA on field, team-season adjusted", "epa_field_adj"),
    ("WR/TE: EPA per target", "epa_target"),
    ("DB: EPA allowed in coverage (flipped)", "epa_coverage"),
    ("DL/EDGE: pressure rate on pass rush", "pressure_rate"),
    ("DL/EDGE: disruption rate on pass rush", "disrupt_rate"),
    ("OL: pressure held rate", "pressure_held"),
    ("DL/EDGE: NFL get-off speed (flipped seconds)", "get_off_speed"),
    ("Game Juice+", "game_juice"),
    ("Game explosiveness", "g_expl_plus"),
    ("Game top speed", "g_top_plus"),
]

# Everything is compared as within-position percentiles, so groups with different scales mix fairly.
for _, c in SCORES + OUTCOMES:
    OD["X_" + c] = OD.groupby("pgm")[c].rank(pct=True)
OD["X_start_share"] = OD.groupby(["pgm", "draft_year"]).start_share.rank(pct=True)

In [ ]:
def corr(x, y):
    return np.corrcoef(x, y)[0, 1]


def resid(v, p):
    X = np.c_[np.ones(len(p)), p]
    return v - X @ np.linalg.lstsq(X, v, rcond=None)[0]


def ci(vals):
    return np.nanpercentile(vals, [2.5, 97.5])


def table_row(xc, yc, with_partial):
    m = OD[["X_" + xc, "X_" + yc, "pick_p"]].notna().all(axis=1)
    x, y, p = (OD.loc[m, "X_" + xc].to_numpy(), OD.loc[m, "X_" + yc].to_numpy(), OD.loc[m, "pick_p"].to_numpy())
    if len(x) < 30:
        return None
    idx = rng_o.integers(0, len(x), (BOOT, len(x)))
    r = corr(x, y)
    lo, hi = ci([corr(x[i], y[i]) for i in idx])
    res = {"n": len(x), "rho": r, "lo": lo, "hi": hi}
    if with_partial:
        res["partial"] = corr(resid(x, p), resid(y, p))
        res["plo"], res["phi"] = ci([corr(resid(x[i], p[i]), resid(y[i], p[i])) for i in idx])
    return res


def edge(a, b, yc):
    m = OD[["X_" + a, "X_" + b, "X_" + yc]].notna().all(axis=1)
    xa, xb, y = (OD.loc[m, "X_" + a].to_numpy(), OD.loc[m, "X_" + b].to_numpy(), OD.loc[m, "X_" + yc].to_numpy())
    if len(y) < 30:
        return None
    idx = rng_o.integers(0, len(y), (BOOT, len(y)))
    d = corr(xa, y) - corr(xb, y)
    return d, *ci([corr(xa[i], y[i]) - corr(xb[i], y[i]) for i in idx]), len(y)

In [ ]:
EDGES = [("juice_final", "ngs_athleticism_score", "Juice+ minus NGS athleticism"),
         ("juice_final", "neg_forty", "Juice+ minus official 40"),
         ("juice_final", "speed_score", "Juice+ minus speed score"),
         ("juice_final", "juice_tracked_only", "Juice+ as built minus tracked-only")]
rows = []
for olabel, oc in OUTCOMES:
    d = OD[OD["X_" + oc].notna()]
    say(f"\n==== {olabel}  (n = {len(d)}; groups: {d.pgm.value_counts().to_dict()})")
    say(f"{'score':38s} {'n':>4s}  corr [95% CI]               | after removing draft pick [95% CI]")
    is_game_outcome = oc in GAME_COLS
    for slabel, sc in SCORES:
        if sc == oc or (is_game_outcome and sc in GAME_COLS):
            continue
        res = table_row(sc, oc, with_partial=(sc != "pick_p"))
        if res is None:
            continue
        part = (f"{res['partial']:+.3f} [{res['plo']:+.3f}, {res['phi']:+.3f}]" if "partial" in res else "n/a")
        say(f"{slabel:38s} {res['n']:4d}  {res['rho']:+.3f} [{res['lo']:+.3f}, {res['hi']:+.3f}]   | {part}")
        rows.append({"outcome": olabel, "score": slabel, **res})
    for a, b, label in EDGES:
        e = edge(a, b, oc)
        if e:
            say(f"  {label}: {e[0]:+.3f} [{e[1]:+.3f}, {e[2]:+.3f}] on the same {e[3]} players")

save(pd.DataFrame(rows), "outcome_test_table.csv")

say("\nPlayers with All-Pro or Pro Bowl votes (Combine Juice+ and pick):")
say(OD[OD.honors][["display_name", "nfl_position", "draft_year", "pick", "juice_final"]]
    .sort_values("juice_final", ascending=False).round(1).to_string(index=False))
say("\nHow to read this: an interval that includes 0 means no clear link. A gap between two scores only"
    " matters if the edge interval excludes 0. Juice+ was built and tuned on the same players as the Game"
    " rows, so any gain there is exploratory. The EPA, coverage, pressure and get-off rows are new evidence.")
(OUT / "outcome_test_report.txt").write_text("\n".join(report_lines))
say(f"\nSaved: {OUT / 'outcome_test_report.txt'}")

## 10. Figures
Each figure is saved as a PNG in the output folder. Colours are colour-blind safe.

In [ ]:
def savefig(fig, name):
    fig.savefig(OUT / name, dpi=200, bbox_inches="tight")
    plt.show()

### Figure 1. Taking body size out of the Combine pieces
Heavier players run the fastest 10 yards of a 40 faster in absolute terms than lighter ones in the same group. The score
compares each player with others of similar size and class, so size alone cannot make a player look elite.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True)
for g, d in FINAL.dropna(subset=["fly10", "fly10_z", "combine_weight"]).groupby("pgm"):
    axes[0].scatter(d.combine_weight, d.fly10 * YD_TO_MPH, s=14, alpha=0.7, color=GROUP_COL[g], label=g)
    axes[1].scatter(d.combine_weight, d.fly10_z, s=14, alpha=0.7, color=GROUP_COL[g])
axes[0].set_title("Before: fastest 10 yards of the 40 (mph)")
axes[1].set_title("After: size- and class-adjusted (z-score)")
for ax in axes:
    ax.set_xlabel("Combine weight (lb)")
axes[0].set_ylabel("top speed (mph)")
axes[1].set_ylabel("standard deviations")
axes[0].legend(ncol=3, fontsize=8, frameon=False)
fig.tight_layout()
savefig(fig, "fig1_size_normalization.png")

### Figure 2. One player, drill vs game (optional)
Set `MAKE_GIF = True` in the settings cell. It re-reads game tracking for a few plays, which takes a few minutes the
first time. `PLAYER` can be any player in the scored set; otherwise the player with the best combined score is used.

In [ ]:
if MAKE_GIF:
    DRILL_SUBS = {"DB": ["BACK_PEDAL"], "WR": ["ROUTE"], "TE": ["ROUTE"], "DL": ["BAG", "PASS_RUSH", "WAVE"],
                  "EDGE": ["BAG", "PASS_RUSH", "WAVE"], "OL": ["PASS_PRO", "MIRROR", "WAVE"]}
    SNAP_WORD = {"DB": "coverage", "WR": "route", "TE": "route", "DL": "pass-rush", "EDGE": "pass-rush", "OL": "pass-pro"}
    row = M[M.display_name == PLAYER]
    if row.empty:
        row = M.assign(_both=np.minimum(M.juice_final, M.game_juice)).nlargest(1, "_both")
    pid, pname, ppos, pg = int(row.nfl_id.iloc[0]), row.display_name.iloc[0], row.nfl_position.iloc[0], row.pgm.iloc[0]
    cj, gj = row.juice_final.iloc[0], row.game_juice.iloc[0]
    gif_path = str(OUT / f"juice_drill_vs_play_{pid}.gif")

    if not os.path.exists(gif_path):
        # Combine: his position drill with the hardest braking and cutting (95th-percentile acceleration)
        mine_d = ad[ad.nfl_id == pid]
        pos_d = mine_d[mine_d.drill_name.apply(lambda n: any(s_ in n for s_ in DRILL_SUBS.get(pg, [])))]
        pick_d = pos_d if len(pos_d) else mine_d
        if len(pick_d):
            ev = pick_d.acc95.idxmax(); dname = pick_d.loc[ev, "drill_name"]
        else:                                             # no usable drill: fall back to his 40
            ev = a40[a40.nfl_id == pid].fly10.idxmax(); dname = "FORTY_YARD_DASH"
        cf = ct[ct.event_id == ev].sort_values("t")
        on = np.where(cf.s.values >= 1.0)[0]
        cf = cf.iloc[max(on[0] - 3, 0):] if len(on) else cf
        cf = cf[cf.t - cf.t.iloc[0] <= 6.0]                 # the drill itself, not the jog out afterwards
        cx, cy = cf.x.values, cf.y.values
        cs, ctt = cf.s.values * YD_TO_MPH, cf.t.values - cf.t.values[0]

        # Game: his sharpest cuts at speed on role-matched snaps (same route as the drill for WR/TE)
        pool = PT[(PT.nfl_id == pid) & PT.role]
        if pg in ("WR", "TE"):
            same = pool[pool.route.astype(str).apply(lambda r: r not in ("NO_ROUTE", "NONE") and r.split("_")[0] in dname)]
            pool = same if len(same) else pool
        cands = pool.nlargest(5, "alat_max")[["game_id", "play_id", "season"]].astype(int)
        if cands.empty:
            raise ValueError(f"No role-matched game snaps for {pname}; set PLAYER to someone else.")
        gp_all = []
        for seas_ in cands.season.unique():     # full pass, no early exit: rows of one play may not be contiguous
            for chunk in pd.read_csv(get_file(f"game_tracking_{seas_}.csv"), chunksize=2_000_000,
                                     usecols=["game_id", "play_id", "nfl_id", "time", "x", "y", "s", "event"]):
                hit = chunk[chunk.game_id.isin(cands.game_id)]
                if len(hit):
                    gp_all.append(hit.merge(cands[["game_id", "play_id"]], on=["game_id", "play_id"]))
        gp_all = pd.concat(gp_all, ignore_index=True)
        gp_all["time"] = pd.to_datetime(gp_all.time)
        me = pd.DataFrame()
        for _, cand in cands.iterrows():
            gp = gp_all[(gp_all.game_id == cand.game_id) & (gp_all.play_id == cand.play_id)].copy()
            mine = gp[gp.nfl_id == pid]
            if mine.empty:
                continue
            snap_t = mine.loc[mine.event == "ball_snap", "time"].min()
            if pd.isna(snap_t):
                snap_t = gp.loc[gp.event == "ball_snap", "time"].min()
            if pd.isna(snap_t):
                snap_t = mine.time.min()
            gp["t"] = (gp.time - snap_t).dt.total_seconds().round(1)
            gp = gp[(gp.t >= 0) & (gp.t <= 6)].sort_values("t")
            me = gp[gp.nfl_id == pid].drop_duplicates("t")
            if len(me) >= 10:
                seas = int(cand.season)
                break
        if len(me) < 10:
            raise ValueError(f"Could not find usable tracking frames for {pname}'s top snaps; set PLAYER to someone else.")
        others = {t: d for t, d in gp[gp.nfl_id != pid].groupby("t")}
        gx, gy, gs, gtt = me.x.values, me.y.values, me.s.values * YD_TO_MPH, me.t.values

        def field(ax, xs, ys, face, line):
            pad = 3
            xc, yc = (xs.min() + xs.max()) / 2, (ys.min() + ys.max()) / 2
            half = max(xs.max() - xs.min(), ys.max() - ys.min()) / 2 + pad
            ax.set_xlim(xc - half, xc + half); ax.set_ylim(yc - half, yc + half); ax.set_aspect("equal")
            ax.set_facecolor(face); ax.set_xticks([]); ax.set_yticks([])
            for v in range(int(np.floor((xc - half) / 5) * 5), int(xc + half) + 1, 5):
                ax.axvline(v, color=line, lw=1.0, zorder=0)
            for v in range(int(np.floor((yc - half) / 5) * 5), int(yc + half) + 1, 5):
                ax.axhline(v, color=line, lw=1.0, zorder=0)

        fig = plt.figure(figsize=(10, 8))
        gsp = fig.add_gridspec(2, 2, height_ratios=[2.2, 1.3], hspace=0.35, wspace=0.08)
        axc, axg, axs = fig.add_subplot(gsp[0, 0]), fig.add_subplot(gsp[0, 1]), fig.add_subplot(gsp[1, :])
        fig.suptitle(f"Figure 2. {pname} ({ppos}): Combine Juice+ {cj:.0f}  |  Game Juice+ {gj:.0f}",
                     x=0.02, ha="left", fontsize=13)
        field(axc, cx, cy, "#eeeeee", "#b5b5b5")
        axc.set_title("Combine drill:\n" + dname.replace("_", " ").title(), loc="left", fontsize=10)
        field(axg, gx, gy, "#dcefdc", "#a9c9a9")
        axg.set_title(f"NFL game ({seas}):\nsharpest cut on a {SNAP_WORD.get(pg, 'game')} snap", loc="left", fontsize=10)
        c_trail, = axc.plot([], [], color=OI[0], lw=2.2); c_dot, = axc.plot([], [], "o", color=OI[0], ms=11)
        o_dots = axg.scatter([], [], s=40, color="grey", alpha=0.6, label="other tracked players")
        g_trail, = axg.plot([], [], color=OI[1], lw=2.2)
        g_dot, = axg.plot([], [], "o", color=OI[1], ms=11, label=pname)
        axg.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, fontsize=8, frameon=False)
        axc.set_xlabel("grid lines every 5 yards in both panels", fontsize=8, color="grey")
        tmax = max(ctt.max(), gtt.max())
        axs.set_xlim(0, tmax); axs.set_ylim(0, max(cs.max(), gs.max()) * 1.15)
        axs.set_xlabel("seconds since he starts moving (Combine) / since the snap (game)"); axs.set_ylabel("speed (mph)")
        c_line, = axs.plot([], [], color=OI[0], lw=2.5, label="Combine drill")
        g_line, = axs.plot([], [], color=OI[1], lw=2.5, ls="--", label="NFL snap")
        axs.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)

        n_frames = int(round(tmax * 10)) + 15          # hold the last frame for 1.5 s
        def frame(i):
            ic, ig = min(i, len(cx) - 1), min(i, len(gx) - 1)
            c_trail.set_data(cx[:ic + 1], cy[:ic + 1]); c_dot.set_data([cx[ic]], [cy[ic]])
            g_trail.set_data(gx[:ig + 1], gy[:ig + 1]); g_dot.set_data([gx[ig]], [gy[ig]])
            od = others.get(gtt[ig])
            o_dots.set_offsets(od[["x", "y"]].values if od is not None else np.empty((0, 2)))
            c_line.set_data(ctt[:ic + 1], cs[:ic + 1]); g_line.set_data(gtt[:ig + 1], gs[:ig + 1])
            return c_trail, c_dot, g_trail, g_dot, o_dots, c_line, g_line
        anim = animation.FuncAnimation(fig, frame, frames=n_frames, interval=100, blit=True)
        anim.save(gif_path, writer=animation.PillowWriter(fps=10))
        plt.close(fig)
    display(Image(filename=gif_path))
else:
    print("Skipped. Set MAKE_GIF = True to build the drill-vs-game animation.")

### Figure 3. Combine Juice+ vs Game Juice+
Each dot is a player. Dot size shows how much game data backs the score (`confidence`); faint dots are thin samples that
are pulled toward 100 by design.

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 5.6))
for g, d in M.groupby("pgm"):
    ax.scatter(d.juice_final, d.game_juice, s=12 + 90 * d.confidence.fillna(0) ** 1.5,
               alpha=0.35 + 0.45 * d.confidence.fillna(0), color=GROUP_COL[g], label=g, edgecolor="none")
ax.axhline(100, color="grey", lw=0.8)
ax.axvline(100, color="grey", lw=0.8)
r_all = STICK[(STICK["sample"] == "everyone scored") & (STICK.position_group == "ALL")].iloc[0]
ax.set_title(f"Rank correlation {r_all.spearman:.2f} (95% CI {r_all.ci95_low:.2f} to {r_all.ci95_high:.2f}), n = {int(r_all.n_players)}",
             loc="left")
ax.set_xlabel("Combine Juice+")
ax.set_ylabel("Game Juice+")
ax.legend(ncol=3, fontsize=8, frameon=False, loc="upper left")
savefig(fig, "fig3_scatter.png")

### Figure 4. How often each Combine band is above average in games
Bars are the share of players in a Combine band whose Game Juice+ is 100 or higher. Whiskers are 95% Wilson intervals.
The dashed line is the rate for all players.

In [ ]:
bt = BANDS.copy()
bt["k"] = (bt.pct_above_avg / 100 * bt.n).round()
ci_b = [wilson(k, n) for k, n in zip(bt.k, bt.n)]
lo = np.array([c[0] for c in ci_b]) * 100
hi = np.array([c[1] for c in ci_b]) * 100
fig, ax = plt.subplots(figsize=(7.4, 4.4))
xs = np.arange(len(bt))
ax.bar(xs, bt.pct_above_avg, color=OI[0], width=0.6)
ax.errorbar(xs, bt.pct_above_avg, yerr=[bt.pct_above_avg - lo, hi - bt.pct_above_avg], fmt="none", ecolor="black", capsize=4)
ax.axhline(100 * M.g_above.mean(), color="grey", ls="--", lw=1)
ax.set_xticks(xs)
ax.set_xticklabels([f"{b}\nn = {n}" for b, n in zip(bt.band, bt.n)], fontsize=9)
ax.set_ylabel("% above average in games")
ax.set_ylim(0, 100)
savefig(fig, "fig4_bands.png")

### Figure 5. Which Combine piece matches which game piece

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 3.9))
im = ax.imshow(CMAT.values.astype(float), cmap="Blues", vmin=0, vmax=max(0.4, np.nanmax(CMAT.values.astype(float))))
ax.set_xticks(range(3))
ax.set_xticklabels(CMAT.columns, fontsize=9)
ax.set_yticks(range(3))
ax.set_yticklabels(CMAT.index, fontsize=9)
for i in range(3):
    for j in range(3):
        v = CMAT.values[i, j]
        ax.text(j, i, "n/a" if not np.isfinite(v) else f"{v:.2f}", ha="center", va="center",
                color="white" if np.isfinite(v) and v > 0.25 else "black")
ax.spines[:].set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="rank correlation")
savefig(fig, "fig5_component_heatmap.png")

### Figure 6. Juice+ against other stopwatch scores
Same players, same game score. Bars are rank correlations with Game Juice+; whiskers are 95% bootstrap intervals.
Overlapping whiskers mean no clear winner.

In [ ]:
if H2H:
    V = H2H["V"]
    cols = {"Juice+": "juice_final", "NGS": "ngs_athleticism_score", "RAS-style": "ras_style", "Official 40": "forty_neg"}
    cols = {k: c for k, c in cols.items() if c in V.columns}

    def pooled_rho_v(df, col):
        return spearmanr(df.groupby("pgm")[col].rank(pct=True), df.groupby("pgm").game_juice.rank(pct=True),
                         nan_policy="omit")[0]

    rb = np.random.default_rng(11)
    est, lo_, hi_ = [], [], []
    boots = {k: [] for k in cols}
    for _ in range(500):
        b = V.iloc[rb.integers(0, len(V), len(V))]
        for k, c in cols.items():
            boots[k].append(pooled_rho_v(b, c))
    for k, c in cols.items():
        est.append(pooled_rho_v(V, c))
        l_, h_ = np.nanpercentile(boots[k], [2.5, 97.5])
        lo_.append(l_)
        hi_.append(h_)
    est, lo_, hi_ = map(np.array, (est, lo_, hi_))
    fig, ax = plt.subplots(figsize=(7.4, 4.2))
    ax.bar(list(cols), est, color=[OI[0] if k == "Juice+" else "#9aa5b1" for k in cols], width=0.6)
    ax.errorbar(range(len(cols)), est, yerr=[est - lo_, hi_ - est], fmt="none", ecolor="black", capsize=4)
    ax.set_ylabel("rank correlation with Game Juice+")
    ax.set_title(f"Same {len(V)} players", loc="left")
    savefig(fig, "fig6_head_to_head.png")
else:
    print("No head-to-head data.")

### Figure 7. Beyond movement: outcomes by score
Each row is an outcome; each dot is the within-position rank correlation of a pre-draft score with that outcome.
Whiskers are 95% bootstrap intervals. Whiskers crossing zero mean no clear link.

In [ ]:
OUTCOME_TAB = pd.DataFrame(rows)
show = {"Juice+ (as built)": (OI[0], "o"), "Official 40 (faster = better)": (OI[1], "s"),
        "NGS athleticism score": (OI[2], "^"), "Draft pick": (OI[7], "D")}
OT = OUTCOME_TAB[OUTCOME_TAB.score.isin(show)].copy()
outs = [o_ for o_ in OT.outcome.unique()][::-1]
fig, ax = plt.subplots(figsize=(9, 0.55 * len(outs) + 1.8))
for k, (sc, (col, mk)) in enumerate(show.items()):
    d = OT[OT.score == sc].set_index("outcome")
    ys = [outs.index(o_) + (k - 1.5) * 0.16 for o_ in d.index]
    ax.errorbar(d.rho, ys, xerr=[d.rho - d.lo, d.hi - d.rho], fmt=mk, color=col, ms=5, lw=1, capsize=2, label=sc)
ax.axvline(0, color="grey", lw=0.8)
ax.set_yticks(range(len(outs)))
ax.set_yticklabels([o_[:52] for o_ in outs], fontsize=8)
ax.set_xlabel("rank correlation with outcome")
ax.legend(fontsize=8, frameon=False, loc="lower right")
savefig(fig, "fig7_outcomes.png")

### Figure 8. Combine vs game by position group

In [ ]:
sg = STICK[(STICK["sample"] == "everyone scored") & (STICK.position_group != "ALL")].sort_values("spearman")
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.errorbar(sg.spearman, range(len(sg)), xerr=[sg.spearman - sg.ci95_low, sg.ci95_high - sg.spearman],
            fmt="o", color=OI[0], capsize=3)
ax.axvline(0, color="grey", lw=0.8)
ax.set_yticks(range(len(sg)))
ax.set_yticklabels([f"{g} (n = {n})" for g, n in zip(sg.position_group, sg.n_players)])
ax.set_xlabel("rank correlation, Combine vs game")
savefig(fig, "fig8_by_position.png")

### Figure 9. Why low-snap players sit near 100
Game Juice+ against the number of usable plays. Players with few plays are shrunk toward 100; the funnel narrows to
the true spread as evidence accumulates.

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.8))
for g, d in GAME.groupby("pgm"):
    ax.scatter(d.npl, d.game_juice, s=14, alpha=0.6, color=GROUP_COL[g], label=g, edgecolor="none")
ax.set_xscale("log")
ax.axhline(100, color="grey", lw=0.8)
ax.set_xlabel("usable plays (log scale)")
ax.set_ylabel("Game Juice+")
ax.legend(ncol=3, fontsize=8, frameon=False)
savefig(fig, "fig9_shrinkage_funnel.png")

## 11. Player tables
The full table (all scored players, with confidence) is saved to the output folder. Below: late-round and undrafted
players with a Combine Juice+ of 110 or more, with their game scores where they have one.

In [ ]:
TABLE = FINAL.merge(GAME[["nfl_id", "game_juice", "confidence", "npl"]], on="nfl_id", how="left")
PLAYER_TABLE = TABLE[["nfl_id", "display_name", "nfl_position", "pgm", "draft_year", "draft_overall_pick", "juice_final",
                      "E_plus", "T_plus", "A_plus", "juice_tracked_only", "game_juice", "confidence", "npl"]]
save(PLAYER_TABLE, "juice_player_table.csv")
board = TABLE[((TABLE.draft_overall_pick >= LOW_PICK) | TABLE.draft_overall_pick.isna()) & (TABLE.juice_final >= HIGH_JUICE)]
display(board.sort_values("juice_final", ascending=False)[
    ["display_name", "nfl_position", "draft_year", "draft_overall_pick", "juice_final", "T_plus", "game_juice", "confidence"]].round(1))
save(STICK, "combine_to_game_comparison.csv")
save(BANDS, "band_outcomes.csv")
save(ANSWERS, "risk_ratios.csv")
print(f"Everything saved to: {OUT.resolve()}")